
# Day 5 – Memory and RAG

## Project: Sweet Crumbs Home Bakery AI Agent

### Objective
Enhance the existing bakery AI agent by adding conversation memory and a knowledge retrieval system.

### Topics Covered
1. Three types of memory.
2. Conversation memory using thread_id.
3. Checkpointers in LangChain.
4. Embeddings and semantic search.
5. Vector databases using ChromaDB.
6. Document chunking.
7. Retrieval-Augmented Generation (RAG).
8. Integrating memory and RAG into the bakery AI agent.

### Technologies Used
- Python
- LangChain
- Groq LLM
- LangGraph
- ChromaDB
- Google Colab

### Learning Outcomes
Build an AI agent that maintains separate conversations for different customers and retrieves relevant bakery policies before answering questions.



## 1. Three Types of Memory

### 1. Conversation Memory
Stores messages exchanged during the current conversation.

Example: Ravi asks for 2 plum cakes and later says he wants them eggless.

### 2. Customer Memory
Stores information about a customer across multiple conversations.

Example: Ravi usually prefers eggless cakes.

### 3. Business Knowledge
Stores business information, rules and policies.

Example: Delivery is free for orders above Rs 1,000.

### Conclusion
Conversation memory remembers the current chat, customer memory remembers individual customers, and business knowledge stores information that applies to the business.


## 2. Conversation Memory Using LangGraph

### The Problem
A normal chatbot may not remember earlier messages unless we manually pass the conversation history.

When multiple customers use the same agent, their conversations must remain separate.

### The Solution
LangGraph provides a checkpointer to save conversation state.

- `InMemorySaver()` stores conversation history temporarily.
- `thread_id` identifies a particular conversation.
- The same `thread_id` continues the same conversation.
- A different `thread_id` starts a separate conversation.

### Example
- `"ravi"` stores Ravi's conversation.
- `"sravani"` stores Sravani's conversation.

This prevents one customer's conversation from mixing with another customer's conversation.

In [1]:
%pip install -q langchain-groq chromadb

In [2]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
import chromadb

print("All imports successful!")

All imports successful!


In [3]:
from google.colab import userdata

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROQ_API_KEY"),
    temperature=0
)

print("Groq LLM initialized successfully!")

Groq LLM initialized successfully!


In [4]:
response = llm.invoke("Introduce yourself as the Sweet Crumbs bakery assistant.")

print(response.content)

👋 Hi there! I’m **Sweet Crumbs**, your friendly bakery assistant. I’m here to help you explore our delicious treats, answer any questions about our menu, or guide you through placing an order. Whether you’re craving a classic croissant, a decadent cake, or something gluten‑free, I’ve got the scoop on all our goodies. Let me know how I can sweeten your day! 🍰🥐✨


## 2. Conversation Memory using thread_id

An ordinary LLM call does not automatically remember previous messages.

Conversation memory allows an AI agent to maintain the context of a conversation.

### What is a thread_id?

A thread_id is a unique identifier assigned to a conversation.

- The same thread_id continues the previous conversation.
- A different thread_id starts a separate conversation.
- A checkpointer saves the conversation state so it can be retrieved later.

### What is InMemorySaver?

InMemorySaver is a LangGraph checkpointer that stores conversation states in memory.

Note: Its stored conversations are temporary and will be lost when the Colab runtime is restarted.

## 3. Restoring the Bakery Tools

Before adding conversation memory, we need to restore the bakery's existing tools.

The three tools are:

1. get_price – Calculates the price of menu items.
2. is_open – Checks the bakery's opening hours.
3. log_order – Validates and saves customer orders to Google Sheets.

We will reconnect to the existing Sweet Crumbs Orders spreadsheet.

In [5]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain.tools import tool

# Bakery menu
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

# Bakery opening hours
HOURS = {
    "monday": "9 am to 8 pm",
    "tuesday": "9 am to 8 pm",
    "wednesday": "9 am to 8 pm",
    "thursday": "9 am to 8 pm",
    "friday": "9 am to 8 pm",
    "saturday": "9 am to 8 pm",
    "sunday": "10 am to 6 pm"
}

# Connect to our existing Google Sheet
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()

creds, _ = default()
gc = gspread.authorize(creds)

sh = gc.open("Sweet Crumbs Orders")
ws = sh.sheet1

print("Bakery data restored!")
print("Connected to:", sh.title)
print("Spreadsheet:", sh.url)

Bakery data restored!
Connected to: Sweet Crumbs Orders
Spreadsheet: https://docs.google.com/spreadsheets/d/1wU70HtZIA9Wt5tiI6UBUBDRcZUG-Sjdz_B31D1ZTCIg


### 3.1 Pydantic Model for Order Validation

Pydantic helps us validate data before saving it.

Our Order model contains:
- name: Customer's name.
- item: Selected bakery item.
- qty: Number of items.
- date: Pickup date.
- eggless: Whether an eggless version is requested.

The model checks whether the received data follows the defined rules.

In [6]:
class Order(BaseModel):
    """A cake order received from a customer."""

    name: str = Field(description="Customer's first name")

    item: Literal[
        "plum cake",
        "chocolate cake",
        "cupcake"
    ] = Field(description="Menu item")

    qty: int = Field(description="Quantity", ge=1)

    date: str = Field(description="Pickup date")

    eggless: bool = Field(description="Whether eggless is requested")

In [7]:
# Valid order
order1 = Order(
    name="Ravi",
    item="chocolate cake",
    qty=2,
    date="24 December",
    eggless=True
)

print(order1)

name='Ravi' item='chocolate cake' qty=2 date='24 December' eggless=True


In [8]:
# Invalid order: quantity is zero

try:
    order2 = Order(
        name="Ravi",
        item="cupcake",
        qty=0,
        date="24 December",
        eggless=False
    )

except Exception as e:
    print("Invalid order rejected:")
    print(e)

Invalid order rejected:
1 validation error for Order
qty
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal


### 3.2 Testing Order Validation

We tested the Pydantic Order model with two cases:

1. Valid order: An order with quantity 2 was accepted.
2. Invalid order: An order with quantity 0 was rejected.

The validation error confirms that Pydantic enforces the minimum quantity rule (`ge=1`).

**Conclusion:** Pydantic prevents invalid order data from proceeding to the next stage.

## 4. Creating LangChain Tools

Our bakery agent needs three tools:

1. `get_price` – Calculates the total price of an order.
2. `is_open` – Checks the bakery's opening hours.
3. `log_order` – Validates and saves an order to Google Sheets.

We use the `@tool` decorator to convert Python functions into LangChain tools.

These tools allow the AI agent to perform real tasks instead of only generating text.

In [9]:
@tool
def get_price(item: str, qty: int = 1) -> str:
    """
    Get the total price in rupees for a bakery item.

    Args:
        item: Menu item name.
        qty: Quantity required.
    """

    price = MENU.get(item.lower().strip())

    if price is None:
        return f"{item} is not on our menu. Available items: {', '.join(MENU)}"

    total = price * qty

    return f"{qty} x {item} = Rs {total}"

In [10]:
print(get_price.invoke({"item": "chocolate cake", "qty": 2}))
print(get_price.invoke({"item": "cupcake", "qty": 3}))
print(get_price.invoke({"item": "donut", "qty": 1}))

2 x chocolate cake = Rs 1300
3 x cupcake = Rs 240
donut is not on our menu. Available items: plum cake, chocolate cake, cupcake


### 4.3 Opening Hours Tool

The `is_open` tool checks the bakery's opening hours.

It accepts a day of the week and retrieves the corresponding timings from the HOURS dictionary.

If the input is not a valid weekday, it returns an appropriate message.

In [11]:
@tool
def is_open(day: str) -> str:
    """
    Check the bakery's opening hours for a given day.

    Args:
        day: Day of the week.
    """

    hours = HOURS.get(day.lower().strip())

    if hours:
        return f"Open {hours} on {day}."

    return f"{day} is not a valid day of the week."

In [12]:
print(is_open.invoke({"day": "Monday"}))
print(is_open.invoke({"day": "Sunday"}))
print(is_open.invoke({"day": "Holiday"}))

Open 9 am to 8 pm on Monday.
Open 10 am to 6 pm on Sunday.
Holiday is not a valid day of the week.


### 4.4 Google Sheets Order Logging Tool

The `log_order` tool saves customer orders to our Google Sheet.

Before saving:
1. It validates the order using the Pydantic Order model.
2. It calculates the total price using the MENU dictionary.
3. It adds the order details to the spreadsheet.

Invalid orders are rejected and are not saved.

In [13]:
@tool
def log_order(
    name: str,
    item: str,
    qty: int,
    date: str,
    eggless: bool
) -> str:
    """
    Save a confirmed cake order to the Google Sheets Orders sheet.

    Args:
        name: Customer's name.
        item: Bakery item.
        qty: Quantity ordered.
        date: Pickup date.
        eggless: Whether an eggless version is requested.
    """

    # Validate the order
    try:
        order = Order(
            name=name,
            item=item.lower().strip(),
            qty=qty,
            date=date,
            eggless=eggless
        )

    except Exception as e:
        return f"Could not save the order: {e}"

    # Calculate the total price
    total = MENU[order.item] * order.qty

    # Save the order in Google Sheets
    ws.append_row([
        order.name,
        order.item,
        order.qty,
        order.date,
        "yes" if order.eggless else "no",
        total
    ])

    return (
        f"Saved: {order.qty} x {order.item} "
        f"for {order.name} on {order.date}, Rs {total}."
    )

**Note:** The following test saves a real row to the Google Sheets spreadsheet. Run it only once to avoid duplicate test orders.

In [14]:
result = log_order.invoke({
    "name": "Day5 Test",
    "item": "cupcake",
    "qty": 2,
    "date": "30 December",
    "eggless": False
})

print(result)

Saved: 2 x cupcake for Day5 Test on 30 December, Rs 160.


### 4.5 Testing Invalid Orders

We test the order logging tool with invalid inputs.

The tool should reject invalid orders and return an error message without saving them to Google Sheets.

In [15]:
# Test 1: Quantity is zero
result1 = log_order.invoke({
    "name": "Test Customer",
    "item": "cupcake",
    "qty": 0,
    "date": "30 December",
    "eggless": False
})

print("Test 1:", result1)


# Test 2: Item is not available
result2 = log_order.invoke({
    "name": "Test Customer",
    "item": "donut",
    "qty": 2,
    "date": "30 December",
    "eggless": False
})

print("Test 2:", result2)

Test 1: Could not save the order: 1 validation error for Order
qty
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal
Test 2: Could not save the order: 1 validation error for Order
item
  Input should be 'plum cake', 'chocolate cake' or 'cupcake' [type=literal_error, input_value='donut', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/literal_error


## 5. Conversation Memory in AI Agents

Conversation memory allows an AI agent to remember previous messages in a conversation.

For example:

Customer: Hi, my name is Ravi.

Customer: What is the price of two chocolate cakes?

Customer: Can you make them eggless?

The agent should remember that the customer's name is Ravi and that the order concerns two chocolate cakes.

### Important Concepts

1. Checkpointer: Saves the conversation state.
2. InMemorySaver: Stores conversation state temporarily in memory.
3. Thread ID: Identifies a particular conversation.

Each customer conversation can have a separate thread ID.

### 5.1 Creating a Checkpointer

LangGraph provides checkpointers to save an agent's conversation state.

We use `InMemorySaver()` to store conversations temporarily in the notebook's memory.

The saved conversations remain available while the Colab runtime is active. They are not permanent database storage.

In [16]:
from langgraph.checkpoint.memory import InMemorySaver

# Create conversation memory
memory = InMemorySaver()

print("Conversation memory created successfully!")

Conversation memory created successfully!


### 5.2 Connecting Memory to the Agent

We attach the checkpointer to our LangChain agent.

The agent now has:
- Brain: Groq LLM.
- Instructions: System prompt.
- Tools: Price, opening hours and order logging.
- Memory: InMemorySaver checkpointer.

The agent can now save and resume conversations using thread IDs.

In [17]:
from google.colab import userdata
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=userdata.get("GROQ_API_KEY"),
    temperature=0
)

print("Groq model restored successfully!")

Groq model restored successfully!


In [18]:
response = llm.invoke("Say hello to Sweet Crumbs.")

print(response.content)

Hello, Sweet Crumbs! 🌟 How can I help you today?


In [19]:
SYSTEM = """
You are the friendly AI assistant for Sweet Crumbs Home Bakery.

RULES:
1. Use get_price for all menu price questions.
2. Use is_open for opening-hours questions.
3. Before saving an order, collect the customer's name, item,
   quantity and pickup date.
4. Use log_order only after collecting all required details
   and receiving confirmation from the customer.
5. Never invent menu items, prices or business policies.
6. If you do not know something, say that you will check
   with the bakery owner.
7. Keep responses short, friendly and helpful.
"""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM,
    checkpointer=memory
)

print("Bakery AI agent created with conversation memory!")

Bakery AI agent created with conversation memory!


### 5.3 Testing Conversation Memory

We use a thread ID to identify a conversation.

When the same thread ID is used for multiple messages, the agent can access previous messages from that conversation.

Different thread IDs represent separate conversations.

We will test whether the agent remembers a customer's name.

In [20]:
# Identify Ravi's conversation
config = {
    "configurable": {
        "thread_id": "ravi"
    }
}

# First message
response1 = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Ravi."}]},
    config
)

print(response1["messages"][-1].content)

Hello Ravi! 👋 What would you like to order from Sweet Crumbs Home Bakery today? If you let me know the item, quantity, and pickup date, I’ll take care of the rest.


In [21]:
# Continue the same conversation
response2 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config
)

print(response2["messages"][-1].content)

Your name is Ravi.


In [22]:
# Create a separate conversation for Sravani
config_sravani = {
    "configurable": {
        "thread_id": "sravani"
    }
}

response3 = agent.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Sravani."}]},
    config_sravani
)

print(response3["messages"][-1].content)

Hi Sravani! 👋 What would you like to order from Sweet Crumbs Home Bakery today? If you let me know the item, how many you’d like, the pickup date, and whether you’d prefer an egg‑less version, I can confirm the details for you.


In [23]:
# Ask Sravani's name in the same conversation
response4 = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config_sravani
)

print(response4["messages"][-1].content)

Your name is Sravani.


### 5.5 Results of Conversation Memory Testing

We tested conversation memory using two customers.

**Test 1: Ravi**
- Thread ID: ravi
- First message: My name is Ravi.
- Second message: What is my name?
- Result: The agent correctly remembered Ravi.

**Test 2: Sravani**
- Thread ID: sravani
- First message: My name is Sravani.
- Second message: What is my name?
- Result: The agent correctly remembered Sravani.

**Conclusion:**
LangGraph checkpointers maintain separate conversation states using thread IDs. This allows multiple customers to interact with the same AI agent without mixing their conversations.

## 6. Understanding Retrieval-Augmented Generation (RAG)

RAG stands for Retrieval-Augmented Generation.

It allows an AI agent to retrieve relevant information from a knowledge database before generating an answer.

### Three Stages of RAG

1. Retrieve: Search the knowledge database for relevant information.
2. Augment: Add the retrieved information to the LLM's context.
3. Generate: Generate an answer using the retrieved information.

### 6.1 Embeddings

Embeddings are numerical representations of text that capture its meaning.

An embedding model converts text into numerical vectors.

Texts with similar meanings generally have nearby vectors, even if they use different words.

### 6.2 Vector Database – ChromaDB

ChromaDB is a vector database used to store text and its embeddings.

It searches for information based on semantic similarity.

ChromaDB retrieves relevant information, while the LLM generates the final answer.

### 6.3 Document Chunking

Chunking is the process of dividing large documents into smaller text segments called chunks.

Each chunk should contain one meaningful idea or topic.

For the Sweet Crumbs project, one FAQ can be stored as one chunk.

Good chunking helps retrieve relevant information without returning an entire large document.

In [24]:
%pip install -q chromadb

In [25]:
import chromadb

print("ChromaDB imported successfully!")

ChromaDB imported successfully!


In [26]:
import chromadb

print("ChromaDB imported successfully!")

ChromaDB imported successfully!


## 7. Creating a Bakery Knowledge Base

A knowledge base stores useful information about the business.

For Sweet Crumbs Home Bakery, we will store frequently asked questions (FAQs) about delivery, payment, cancellation, eggless cakes and other policies.

We will use ChromaDB to store these FAQs and retrieve relevant information based on the customer's question.

Note: The following FAQ policies are practice data from the workshop tutorial. They are not all confirmed real-world policies of Sweet Crumbs.

In [27]:
PRACTICE_FAQS = {
    "delivery": "We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1,000.",

    "payment": "We accept UPI and cash. Orders above Rs 1,000 need a 50% advance by UPI.",

    "cancellation": "Cancellation is free up to 12 hours before pickup. After that, the advance is not refunded.",

    "photo_cake": "Photo cakes and custom designs need 48 hours' notice.",

    "notice": "Regular orders need 24 hours' notice.",

    "eggless": "All our cakes have eggless versions at the same price.",

    "allergens": "Plum cake contains nuts and dried fruit. Cupcakes and chocolate cake are nut-free.",

    "storage": "Keep cakes in the fridge and eat them within 3 days.",

    "message": "A message on the cake is free, up to 20 letters.",

    "candles": "Candles are free with every cake.",

    "discounts": "Bulk-order discounts need the owner's approval.",

    "pickup": "Pickup is during opening hours: Monday to Saturday 9 am to 8 pm, Sunday 10 am to 6 pm."
}

print("Total FAQs:", len(PRACTICE_FAQS))

Total FAQs: 12


## 8. Storing FAQs in ChromaDB

ChromaDB organizes stored information into collections.

A collection is similar to a table in a database. We will create a collection named `sweet_crumbs_faq` to store our bakery FAQs.

Each FAQ will have:
- An ID to identify it.
- A document containing the actual information.

In [28]:
import chromadb

# Create a ChromaDB client
client = chromadb.Client()

# Create or access the FAQ collection
faq = client.get_or_create_collection(
    name="sweet_crumbs_faq"
)

print("FAQ collection created successfully!")

FAQ collection created successfully!


In [29]:
# Add topic names to the FAQ documents
faq_documents = [
    f"{topic.replace('_', ' ').title()}: {answer}"
    for topic, answer in PRACTICE_FAQS.items()
]

# Store the FAQs in ChromaDB
faq.upsert(
    ids=list(PRACTICE_FAQS.keys()),
    documents=faq_documents
)

print(faq.count(), "FAQs stored successfully!")

12 FAQs stored successfully!


## 9. Performing Semantic Search

Semantic search retrieves information based on the meaning of a query rather than only matching exact keywords.

ChromaDB uses embeddings to compare the customer's question with stored FAQ documents and returns the most relevant results.

In [30]:
results = faq.query(
    query_texts=["Can you send the cake to my house?"],
    n_results=2
)

print("Retrieved FAQs:")

for document in results["documents"][0]:
    print("-", document)

Retrieved FAQs:
- Message: A message on the cake is free, up to 20 letters.
- Photo Cake: Photo cakes and custom designs need 48 hours' notice.


In [31]:
results = faq.query(
    query_texts=["Can you deliver a cake to my home?"],
    n_results=5,
    include=["documents", "distances"]
)

for document, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print("FAQ:", document)
    print("Distance:", distance)
    print("--------------------")

FAQ: Message: A message on the cake is free, up to 20 letters.
Distance: 0.9801840782165527
--------------------
FAQ: Photo Cake: Photo cakes and custom designs need 48 hours' notice.
Distance: 1.0034677982330322
--------------------
FAQ: Delivery: We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1,000.
Distance: 1.0109679698944092
--------------------
FAQ: Storage: Keep cakes in the fridge and eat them within 3 days.
Distance: 1.0449037551879883
--------------------
FAQ: Candles: Candles are free with every cake.
Distance: 1.108071208000183
--------------------


### 9.1 Observations from Semantic Search

We tested semantic search using different ways of asking about cake delivery.

The results showed that ChromaDB can retrieve the delivery FAQ even when the customer's wording is different from the stored document.

However, the delivery FAQ was not always ranked first.

This demonstrates that semantic search is approximate and does not guarantee that the first retrieved document is always the correct answer.

Retrieval quality should be tested and evaluated before using the retrieved information to answer customers.

In [32]:
faq_documents = [
    f"{topic.replace('_', ' ').title()}: {answer}"
    for topic, answer in PRACTICE_FAQS.items()
]

print(faq_documents[0])

Delivery: We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1,000.


In [33]:
faq.upsert(
    ids=list(PRACTICE_FAQS.keys()),
    documents=faq_documents
)

print("FAQ documents updated!")

FAQ documents updated!


In [34]:
results = faq.query(
    query_texts=["Can you deliver a cake to my home?"],
    n_results=3
)

print("Search results:\n")

for document in results["documents"][0]:
    print("-", document)

Search results:

- Message: A message on the cake is free, up to 20 letters.
- Photo Cake: Photo cakes and custom designs need 48 hours' notice.
- Delivery: We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1,000.


## 10. Creating a RAG Search Tool

We will create a LangChain tool called `search_faq`.

This tool will:
1. Accept a customer's question.
2. Search the ChromaDB FAQ collection.
3. Retrieve the three most relevant documents.
4. Return the retrieved information to the AI agent.

The AI agent can then use this information to answer the customer.

In [35]:
from langchain.tools import tool

@tool
def search_faq(question: str) -> str:
    """
    Search the Sweet Crumbs practice FAQ knowledge base.
    Use this tool for questions about delivery, payment,
    cancellation, eggless cakes, allergens and other policies.
    """

    results = faq.query(
        query_texts=[question],
        n_results=3
    )

    documents = results["documents"][0]

    if not documents:
        return "No relevant FAQ information was found."

    return "\n".join(documents)

In [36]:
result = search_faq.invoke({
    "question": "Can you deliver cakes to my home?"
})

print(result)

Storage: Keep cakes in the fridge and eat them within 3 days.
Delivery: We deliver within 5 km of the bakery. Delivery is Rs 50, free for orders above Rs 1,000.
Message: A message on the cake is free, up to 20 letters.


## 11. Integrating RAG with the AI Agent

We will integrate the ChromaDB search tool with our existing bakery AI agent.

The agent will use:
- Python tools for prices, timings and order logging.
- ChromaDB retrieval for bakery policies.
- Conversation memory to remember individual customer conversations.

This creates an AI agent that can remember conversations and retrieve business knowledge.

In [37]:
SYSTEM = """
You are the friendly AI assistant for Sweet Crumbs Home Bakery.

RULES:
1. Use get_price for all menu price questions.
2. Use is_open for opening-hours questions.
3. Use search_faq for questions about bakery policies.
4. Answer policy questions only using information returned by search_faq.
5. If the retrieved information does not answer the question,
   say that you will check with the bakery owner.
6. Before saving an order, collect the customer's name, item,
   quantity and pickup date.
7. Use log_order only after collecting all required details
   and receiving confirmation from the customer.
8. Never invent menu items, prices or business policies.
9. Keep responses short, friendly and helpful.
"""

print("Final RAG system prompt created!")

Final RAG system prompt created!


In [38]:
agent = create_agent(
    model=llm,
    tools=[
        get_price,
        is_open,
        log_order,
        search_faq
    ],
    system_prompt=SYSTEM,
    checkpointer=memory
)

print("Sweet Crumbs RAG Agent created successfully!")

Sweet Crumbs RAG Agent created successfully!


In [39]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order, search_faq],
    system_prompt=SYSTEM,
    checkpointer=memory
)

print("Final Sweet Crumbs RAG Agent ready!")


Final Sweet Crumbs RAG Agent ready!


In [40]:
config = {"configurable": {"thread_id": "day5_test"}}

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "Do you deliver cakes to homes?"}
    ]},
    config
)

print(response["messages"][-1].content)

Yes! We deliver cakes within a 5 km radius of the bakery. Delivery costs Rs 50, but it’s free for orders over Rs 1,000. Let me know if you’d like to place an order or need more details.


In [41]:
response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "What is my name?"}
    ]},
    config
)

print(response["messages"][-1].content)

I don’t have that information. Could you let me know your name?


In [42]:
# Create a separate conversation for the final test
config = {
    "configurable": {
        "thread_id": "day5_test"
    }
}

# First message: introduce the customer
agent.invoke(
    {"messages": [
        {"role": "user", "content": "Hi, my name is Alex."}
    ]},
    config
)

# Second message: check whether the agent remembers
response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "What is my name?"}
    ]},
    config
)

print(response["messages"][-1].content)

Your name is Alex.


## 12. Conclusion and Learning Outcomes

In Day 5, we enhanced the Sweet Crumbs AI agent with conversation memory and Retrieval-Augmented Generation (RAG).

### What I Learned
1. Different types of memory used in AI agents.
2. Conversation memory using LangGraph checkpointers.
3. Embeddings and semantic search.
4. Storing and retrieving information using ChromaDB.
5. Creating a RAG search tool using LangChain.
6. Integrating memory and RAG into an AI agent.

### Final Outcome
Built an AI bakery assistant that can:
- Remember individual customer conversations.
- Retrieve relevant information from a FAQ knowledge base.
- Answer policy questions using retrieved information from the practice FAQ knowledge base.
- Check menu prices and opening hours.
- Validate and log customer orders when Google Sheets is connected.
- Maintain separate conversation histories for different customers using thread IDs.

### Future Improvements
- Improve semantic retrieval accuracy.
- Add verified business policies.
- Expand the knowledge base with additional documents.
- Improve customer and order management.